# 08E – Robustness & Stress Testing

Enterprise notebook for evaluating how stable the bankruptcy prediction model remains under noisy and perturbed input data.

## Business Objective
Assess the robustness of the production model by introducing controlled perturbations into the cleaned dataset and measuring performance degradation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [ ]:
DATA_PATH='american_bankruptcy_cleaned.csv'
MODEL_PATH='production_bankruptcy_model.joblib'

df = pd.read_csv(DATA_PATH)

if 'status_label' in df.columns:
    y = df['status_label'].map({'alive':0,'failed':1})
    X = df.drop(columns=['status_label'])
elif 'target' in df.columns:
    y = df['target']
    X = df.drop(columns=['target'])
else:
    raise ValueError('Target column not found')

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = joblib.load(MODEL_PATH)


In [ ]:
# Baseline performance
baseline_pred = model.predict(X_test)
baseline_prob = model.predict_proba(X_test)[:,1]

baseline = {
    'Accuracy': accuracy_score(y_test, baseline_pred),
    'F1': f1_score(y_test, baseline_pred),
    'ROC_AUC': roc_auc_score(y_test, baseline_prob)
}
baseline

In [ ]:
# Add controlled Gaussian noise to numeric features
X_noisy = X_test.copy()

numeric_cols = X_noisy.select_dtypes(include='number').columns

for col in numeric_cols:
    std = X_noisy[col].std()
    noise = np.random.normal(0, 0.05 * std, len(X_noisy))
    X_noisy[col] = X_noisy[col] + noise

noisy_pred = model.predict(X_noisy)
noisy_prob = model.predict_proba(X_noisy)[:,1]

robustness = pd.DataFrame({
    'Metric':['Accuracy','F1','ROC_AUC'],
    'Baseline':[
        baseline['Accuracy'],
        baseline['F1'],
        baseline['ROC_AUC']
    ],
    'Noisy_Data':[
        accuracy_score(y_test,noisy_pred),
        f1_score(y_test,noisy_pred),
        roc_auc_score(y_test,noisy_prob)
    ]
})

robustness['Performance_Drop'] = robustness['Baseline'] - robustness['Noisy_Data']
robustness.to_csv('robustness_results.csv',index=False)
robustness

In [ ]:
plt.figure(figsize=(8,5))
x=np.arange(len(robustness))
width=0.35

plt.bar(x-width/2, robustness['Baseline'], width, label='Baseline')
plt.bar(x+width/2, robustness['Noisy_Data'], width, label='Noisy')

plt.xticks(x, robustness['Metric'])
plt.ylabel('Score')
plt.title('Robustness Stress Test')
plt.legend()
plt.tight_layout()
plt.savefig('robustness_stress_test.png', dpi=300)
plt.show()

## Business Interpretation

- Small performance drops indicate that the model is robust to minor variations in financial data.
- Large drops suggest the model may be sensitive to noise, requiring additional regularization, feature engineering, or retraining.
- Robustness testing is important for production systems where input data quality can vary over time.

## Deliverables

- `robustness_results.csv`
- `robustness_stress_test.png`

This notebook demonstrates that the bankruptcy prediction model has been evaluated for stability under realistic data perturbations.